# Replace exchange-correlation by Dirac exchange.

This notebook uses a QEpy **iterative** `Driver` to run a custom exchange-correlation potential, the Dirac (Slater) exchange functional, which shows that QEpy provides a useful API for machine learned or any custom functional.

In [ ]:
# --- optional pip installs (normally leave collapsed / do not run) ---
!pip install qepy f90wrap==0.2.16
!pip install matplotlib

## 1) Imports

NumPy for the exchange formula, and QEpy's `Driver`/`QEInput` for running SCF. (The ASE/matplotlib imports below are unused in this minimal demo — left over from a template.)

In [2]:
import numpy as np
from qepy.driver import Driver
from qepy.io import QEInput
from ase.io.trajectory import Trajectory
from ase import Atoms
import matplotlib.pyplot as plt

## 2) System: bulk aluminum

FCC aluminum, one atom per primitive cell, described with an ONCV PBE pseudopotential. `degauss`/`occupations: smearing` handle the metallic Fermi surface; `ecutwfc=30` Ry and a 4×4×4 k-mesh keep this quick to run.

In [3]:
qe_options = {
    '&control': {
        'calculation': "'scf'",
        'pseudo_dir': "'./data/'"
    },
    '&system': {
        'ibrav' : 0,
        'degauss': 0.005,
        'ecutwfc': 30,
        'nat': 1,
        'ntyp': 1,
        'occupations': "'smearing'"
    },
    'atomic_positions crystal': ['Al    0.0  0.0  0.0'],
    'atomic_species': ['Al  26.98 Al_ONCV_PBE-1.2.upf'],
    'k_points automatic': ['4 4 4 0 0 0'],
    'cell_parameters angstrom':[
        '0.     2.025  2.025',
        '2.025  0.     2.025',
        '2.025  2.025  0.   '],
}

## 3) Iterative driver for a custom XC

Create a `Driver` with `iterative=True`. This hands the SCF loop to Python — instead of `driver.scf()` running everything internally, we now call `diagonalize()`/`mix()` ourselves each cycle and can inject a custom potential via `set_external_potential` before each step.

In [4]:
driver=Driver(qe_options=qe_options, iterative = True, logfile='tmp.out')

## 5) Dirac (Slater) exchange functional

`diracx` implements the analytic LDA exchange potential and energy for a given density `rho`:

- $V_x(r) = -\left(\frac{3}{\pi}\rho(r)\right)^{1/3}$
- $E_x = \frac{3}{4}\int V_x(r)\,\rho(r)\,dr$

The `* 2` factors convert from Hartree to Rydberg units, matching QEpy's internal convention (the same convention used by `set_external_potential`).

In [5]:
def dirac_xc(driver):
    # Dirac exchange
    # factor 2 is Ha to Ry
    volume = driver.get_volume()
    rho = driver.get_density()
    dr = volume / rho.size
    v = -np.cbrt(3.0 / np.pi * rho)
    e = np.sum(v * rho) * (3.0 / 4.0) * dr
    return v * 2, e * 2

## 6) Manual SCF loop using Dirac exchange only

Each iteration: get the current density and cell volume, evaluate the Dirac exchange potential/energy, feed it in via `set_external_potential(exttype='xc')` — which **replaces** the pseudopotential's built-in XC entirely, leaving pure exchange with no correlation — then `diagonalize()` and `mix()`. Stop early once `check_convergence()` is satisfied (cap at 60 iterations).

In [7]:
for i in range(60):
    extpot, ex = dirac_xc(driver)
    driver.set_external_potential(potential=extpot, extene=ex, exttype=('xc'))
    driver.diagonalize()
    driver.mix()
    converged = driver.check_convergence()
    print ('Iter: ',i,' - Conv: ', driver.get_scf_error())
    if converged : break

Iter:  0  - Conv:  0.08525245066361664
Iter:  1  - Conv:  0.001173812464203942
Iter:  2  - Conv:  3.13300482496171e-05
Iter:  3  - Conv:  6.047646929723297e-08
